In [82]:
import cv2
import numpy as np
import open3d as o3d
import open3d.visualization as vis

In [83]:
distance_threshold = 0.01 # трешхолд для поиска плоскости пола RANSAC
grid_step = 0.002 # размер клетки в метрах при разбиении пола на сетку
max_dist = 0.4 # высота от пола, на которой точки считаются припятствованиями

In [84]:
def project_to_plane(points, plane_model):
    [a, b, c, d] = plane_model
    points = np.asarray(points)
    distances = a * points[:, 0] + b * points[:, 1] + c * points[:, 2] + d

    projected_points = points - np.outer(distances, normal)
    return projected_points


def switch_to_plane_coords(projected):
    dot_v1 = np.dot(projected, plane_v1)
    dot_v2 = np.dot(projected, plane_v2)

    plane_coords = np.column_stack((dot_v1, dot_v2))
    x_max, x_min = np.max(plane_coords[:, 0]), np.min(plane_coords[:, 0])
    y_max, y_min = np.max(plane_coords[:, 1]), np.min(plane_coords[:, 1])
    return plane_coords, x_max, x_min, y_max, y_min


def visualise(np_arr):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np_arr)
    vis.draw_geometries([pcd])

In [ ]:
pcd_obj = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(pcd_obj.path)

plane_model, inliers = pcd.segment_plane(distance_threshold = distance_threshold,
                                         num_iterations = 10000,
                                         ransac_n = 3)
[a, b, c, d] = plane_model

#смотрим че там у нас за плоскость такая
inlier_points = pcd.select_by_index(inliers)
inlier_points.paint_uniform_color([0, 1, 0])
outlier_points = pcd.select_by_index(inliers, invert = True)
outlier_points.paint_uniform_color([1, 0, 0])
vis.draw_geometries([inlier_points, outlier_points])

#строим нормаль к поверхности   
norm = np.linalg.norm([a, b, c])
normal = np.array([a, b, c]) / norm

# берем 2 перпендикулярных вектора в плоскости
v1_norm = np.linalg.norm([b, -a, 0])
plane_v1 = np.asarray([b, -a, 0]) / v1_norm
plane_v2 = np.linalg.cross(normal, plane_v1)

#проецируем точки пола на плоскость пола
projected = project_to_plane(np.asarray(inlier_points.points), plane_model=plane_model)
visualise(projected)

#переходим от координат в пространстве в координаты 2-х векторов в плоскости
plane_coords, x_max, x_min, y_max, y_min = switch_to_plane_coords(projected)

#строим двумерный массив, соответствующий плоскости
grid = np.zeros((
    int((y_max - y_min) // grid_step + 1), 
    int((x_max - x_min) // grid_step + 1)
))
x_indices = ((plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((plane_coords[:, 1] - y_min) // grid_step).astype(int)
valid_mask = (x_indices >= 0) & (x_indices < grid.shape[1]) & \
             (y_indices >= 0) & (y_indices < grid.shape[0])

grid[y_indices[valid_mask], x_indices[valid_mask]] = 1

# смотрим че получилось
grid = cv2.morphologyEx(grid, cv2.MORPH_CLOSE, np.ones((9, 9)))
grid = cv2.morphologyEx(grid, cv2.MORPH_OPEN, np.ones((7, 7)))
cv2.imshow('Floor, картина в цвете', (grid * 255).astype(np.uint8))
cv2.waitKey(0)
cv2.destroyAllWindows()

# берем только точки близкие к полу
outliers = np.asarray(outlier_points.points)
condition = abs(a*outliers[:, 0] + b*outliers[:, 1] + c*outliers[:, 2] + d) <= max_dist
close_outliers = outliers[condition]
visualise(close_outliers)

# проецируем на плоскость
projected_outliers = project_to_plane(outliers, plane_model=plane_model)
visualise(projected_outliers)

# убираем из плоскости точки, где есть препятствия
outliers_plane_coords = switch_to_plane_coords(projected_outliers)[0]
x_indices = ((outliers_plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((outliers_plane_coords[:, 1] - y_min) // grid_step).astype(int)
valid_mask = (x_indices >= 0) & (x_indices < grid.shape[1]) & \
             (y_indices >= 0) & (y_indices < grid.shape[0])

grid[y_indices[valid_mask], x_indices[valid_mask]] = 0

# смотрим результат
grid = cv2.morphologyEx(grid, cv2.MORPH_OPEN, np.ones((9, 9)))
grid = cv2.morphologyEx(grid, cv2.MORPH_CLOSE, np.ones((7, 7)))
cv2.imshow('Floor with obstacles', (grid * 255).astype(np.uint8))
cv2.waitKey(0)
cv2.destroyAllWindows()

grid shape = (873, 1411)
